# B4 jeepney RL baseline

Traffic-biased quadrilateral baselines on the physical drivable street network.
The minimum-area rule is inspired by polygonal morphology bounds discussed in
https://arxiv.org/html/2603.28385v1.


## 1. Traffic-Biased Quadrilateral Baseline Generator

Builds traffic-weighted quadrilateral routes on the physical drive graph, then stitches the shortest physical path between the four selected anchors.

In [1]:
import pandas as pd
import secrets
from pathlib import Path
from types import SimpleNamespace

from IPython.display import IFrame, display
from tqdm.auto import tqdm

from _bootstrap import ROOT
from utils import BaselineRoute, BaselineRouteGenerator, JeepneySystem, JeepneyRouteEnv, SystemicFitnessEvaluator, calculate_route_fitness, train_route_agent

OUTPUT_DIR = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis") / "results" / "B4_baseline_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROUTES = 20
random_seed = secrets.randbits(32)
generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=random_seed,
)

routes = generator.generate_routes(NUM_ROUTES, route_prefix="B4", seed=random_seed)
route_system = JeepneySystem.from_routes(routes)

summary = pd.DataFrame(
    [
        {
            "route_id": route.route_id,
            "anchors": list(route.ordered_anchor_node_ids),
            "area_m2": round(route.polygon_area_m2, 0),
            "length_m": round(route.path_length_m, 0),
            "attempt": route.attempt_index,
        }
        for route in routes
    ]
)
summary


,route_id,anchors,area_m2,length_m,attempt
0,B401,"[1018, 1406, 908, 3692]",69703128.0,61587.0,1
1,B402,"[1071, 1240, 101, 1685]",6845578.0,26414.0,1
2,B403,"[1681, 129, 1090, 3124]",5975720.0,18489.0,1
3,B404,"[36, 1795, 1351, 771]",5050245.0,24525.0,1
4,B405,"[557, 217, 473, 1090]",5466196.0,17740.0,1
5,B406,"[2828, 34, 1685, 1240]",29434398.0,42589.0,1
6,B407,"[795, 1098, 641, 3516]",7122488.0,46172.0,1
7,B408,"[1680, 703, 1003, 88]",32894460.0,35824.0,2
8,B409,"[177, 30, 197, 381]",24729861.0,43637.0,1
9,B410,"[3197, 997, 4, 698]",19265728.0,82480.0,1


## 2. RL Environment and Coordinate-Invariant Geometric Embeddings

Uses only the primal drivable network to build route geometry step by step, while encoding turn structure, sinuosity, and origin-relative features without absolute node IDs.

In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
env = JeepneyRouteEnv(
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    seed=random_seed,
    max_steps=12,
)

def print_observation(obs: dict[str, np.ndarray]) -> None:
    print('shape:', obs['shape'])
    print('history:', obs['history'])
    print('topology:', obs['topology'])
    print('demand:', obs['demand'])
    print('global:', obs['global'])
    print('candidates:')
    for candidate_index, row in enumerate(obs['candidates']):
        print(f'  {candidate_index}: {row}')
    print('mask:', obs['action_mask'])

obs, info = env.reset()
print('reset state vector length:', info['state_vector'].shape[0])
print_observation(obs)

rng = np.random.default_rng(random_seed)
for step_index in range(5):
    valid_actions = np.flatnonzero(obs['action_mask'][:-1])
    action = int(rng.choice(valid_actions)) if len(valid_actions) else env.max_candidates
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'step {step_index + 1}: action={action}, reward={reward:.3f}, terminated={terminated}, truncated={truncated}')
    print('turn angle rad:', info['turn_angle_rad'])
    print('sinuosity index:', info['sinuosity_index'])
    print('distance to origin m:', info['distance_to_origin_m'])
    print('bearing to origin rad:', info['bearing_to_origin_rad'])
    print('route area m2:', info['route_area_m2'])
    print('state vector:', info['state_vector'])
    print_observation(obs)
    if terminated or truncated:
        break


reset state vector length: 91
shape: [0. 0. 1. 0. 0. 0. 0.]
history: [0. 0. 0. 0. 0. 0. 0. 0.]
topology: [0.5   0.5   0.    0.5   0.556]
demand: [0.226 0.357 0.511 0.285]
global: [0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.5   0.5   0.    0.5   0.556 0.226 0.357 0.511 0.285]
candidates:
  0: [-0.389 -0.921  0.011  0.667  0.     0.511  0.285  0.     1.     0.   ]
  1: [-0.295  0.956  0.017  0.5    0.     0.28   0.053  0.     0.     0.   ]
  2: [0.618 0.786 0.009 0.5   0.    0.28  0.053 0.    0.    0.   ]
  3: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  4: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  5: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
mask: [1. 1. 1. 0. 0. 0. 0.]
step 1: action=1, reward=0.250, terminated=False, truncated=False
turn angle rad: 0.0
sinuosity index: 1.0
distance to origin m: 117.46934991757044
bearing to origin rad: 3.141592653589793
route area m2: 0.0
state vector: [ 0.002  0.    -1.     0.002  0.002  0.     0.     0.     0.     0.
  0.     1.   

In [3]:
route_edges = pd.DataFrame(
    [(int(u), int(v), "start_walk") for u, v in generator.drive_graph_raw.edges()],
    columns=["u", "v", "edge_type"],
)
route_manager = SimpleNamespace(
    edges=route_edges,
    _node_coords={
        int(row.base_node_id): (float(row.lat), float(row.lon))
        for row in generator.node_table.itertuples(index=False)
    },
)

def build_route_encoding(route_id: str) -> str:
    return f"route_id: {route_id}"

def interpret_embeddings(route: BaselineRoute) -> str:
    """Generate human-readable interpretations of route geometry."""
    route_path = route.path_node_ids
    if not route_path or len(route_path) < 2:
        return "Route too short to analyze."
    
    lines = []
    coords = route.path_latlon
    
    area = route.polygon_area_m2
    length = route.path_length_m
    area_to_length = area / max(length, 1.0)
    
    if area_to_length > 1000:
        lines.append("shape: Compact, efficient use of space")
    elif area_to_length > 200:
        lines.append("shape: Moderate coverage relative to distance")
    else:
        lines.append("shape: Linear route with extended length")
    
    node_count = len(route_path)
    if node_count > 50:
        lines.append("topology: High node density, complex route structure")
    elif node_count > 25:
        lines.append("topology: Moderate complexity with multiple waypoints")
    else:
        lines.append("topology: Simple, direct route path")
    
    if area > 50_000_000:
        lines.append("demand: Large service area, potentially high demand coverage")
    elif area > 10_000_000:
        lines.append("demand: Medium service area with moderate demand potential")
    else:
        lines.append("demand: Compact service area with focused demand")
    
    if len(coords) >= 2:
        import math
        start = coords[0]
        end = coords[-1]
        straight_dist = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2) * 111000
        actual_path = length
        sinuosity = actual_path / max(straight_dist, 1.0)
        
        if sinuosity > 1.5:
            lines.append("history: Winding route with many turns")
        elif sinuosity > 1.1:
            lines.append("history: Moderate turning pattern")
        else:
            lines.append("history: Direct path with minimal deviations")
    
    return chr(10).join(lines) if lines else "No embedding data available."

def format_route_fitness(result) -> str:
    return (
        f"standalone fitness: reward={result.reward:.3f} | avg_gtc={result.average_gtc:.3f} | "
        f"passenger_std={result.passenger_gtc_std:.3f}"
    )

def format_systemic_fitness(result) -> str:
    return (
        f"systemic fitness: avg_gtc={result.average_gtc:.3f} ± {result.std_gtc:.3f} | "
        f"avg_passenger_std={result.average_passenger_gtc_std:.3f} ± {result.std_passenger_gtc_std:.3f}"
    )

NOTE_SYSTEMIC_EVAL_TEST_MEAN = 3  # Lightweight note-time sample; increase later for final analysis.
NOTE_SYSTEMIC_EVAL_TEST_STD = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD = 0
FULL_SYSTEMIC_EVAL_TEST_MEAN = 10  # Temporary placeholder; increase later for final sensitivity analysis.
FULL_SYSTEMIC_EVAL_TEST_STD = 5
FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 2
FULL_SYSTEMIC_BACKGROUND_ROUTE_STD = 1

note_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=NOTE_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=NOTE_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)
full_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=FULL_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=FULL_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=FULL_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)

standalone_fitness_cache: dict[tuple[int, ...], object] = {}
systemic_fitness_cache: dict[tuple[int, ...], object] = {}

def route_signature(route) -> tuple[int, ...]:
    return tuple(int(node_id) for node_id in route.path_node_ids)

def get_standalone_fitness(route):
    key = route_signature(route)
    cached = standalone_fitness_cache.get(key)
    if cached is None:
        cached = calculate_route_fitness(
            route.path_node_ids,
            passenger_map=generator.passenger_map,
            drive_graph_raw=generator.drive_graph_raw,
            drive_graph_proj=generator.drive_graph_proj,
            seed=random_seed,
            batch_size=5,
        )
        standalone_fitness_cache[key] = cached
    return cached

def get_systemic_fitness(route, *, use_full: bool = False):
    key = route_signature(route)
    cache = systemic_fitness_cache
    cached = cache.get((key, use_full))
    if cached is None:
        evaluator = full_systemic_evaluator if use_full else note_systemic_evaluator
        cached = evaluator.evaluate(route, seed=random_seed)
        cache[(key, use_full)] = cached
    return cached

def build_route_nodes(summary_df: pd.DataFrame, routes: list) -> dict[str, dict[str, str]]:
    notes: dict[str, dict[str, str]] = {}
    route_by_id = {r.route_id: r for r in routes}
    total = len(summary_df)
    if total == 0:
        return notes
    
    for row in tqdm(summary_df.itertuples(index=False), total=total, desc="Building route notes"):
        route = route_by_id.get(row.route_id)
        interpretation = interpret_embeddings(route) if route else "Route data unavailable."
        if route is not None:
            standalone = get_standalone_fitness(route)
            systemic = get_systemic_fitness(route)
            interpretation = (
                f"{format_route_fitness(standalone)}\n"
                f"{format_systemic_fitness(systemic)}\n\n"
                f"{interpretation}"
            )
        notes[row.route_id] = {
            "encoding": build_route_encoding(row.route_id),
            "interpretation": interpretation,
        }
    return notes

route_notes = build_route_nodes(summary, routes)


Building route notes:   0%|          | 0/20 [00:00<?, ?it/s]

## 3. 3-Layer Graph Reward Formulation
Connect the generated physical shape to the 3-layer behavioral passenger graph to calculate the fitness score (Reward).

In [4]:
comparison_generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=1,
)
comparison_routes = comparison_generator.generate_routes(2, route_prefix='CMP', seed=1)
good_shape = comparison_routes[0].path_node_ids
bad_shape = comparison_routes[1].path_node_ids

good_result = calculate_route_fitness(
    good_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)
bad_result = calculate_route_fitness(
    bad_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)

comparison = pd.DataFrame(
    [
        {"route_shape": "good", "reward": round(good_result.reward, 3), "avg_gtc": round(good_result.average_gtc, 3), "unserved": good_result.unserved_passenger_count},
        {"route_shape": "bad", "reward": round(bad_result.reward, 3), "avg_gtc": round(bad_result.average_gtc, 3), "unserved": bad_result.unserved_passenger_count},
    ]
)
print(f"Good route reward: {good_result.reward:.3f}")
print(f"Bad route reward: {bad_result.reward:.3f}")
comparison


Good route reward: -174.980
Bad route reward: -176.625


,route_shape,reward,avg_gtc,unserved
0,good,-174.980,174.980,0
1,bad,-176.625,176.625,0


## 4. Systemic Synergy Evaluation Utility
Build a statistical evaluation utility that tests a candidate route across multiple background networks to measure true system synergy.

In [5]:
systemic_result = full_systemic_evaluator.evaluate(routes[0], seed=random_seed)
print(f"Systemic average GTC: {systemic_result.average_gtc:.3f}")
print(f"Systemic GTC std dev: {systemic_result.std_gtc:.3f}")
print(f"Passenger GTC std avg: {systemic_result.average_passenger_gtc_std:.3f}")
print(f"Tests used: {systemic_result.n_tests}")


Systemic average GTC: 6152.394
Systemic GTC std dev: 1312.061
Passenger GTC std avg: 14689.644
Tests used: 14


## For Evaluation: Export HTML Tester

In [6]:
html_path = route_system.export_route_toggle_html(
    route_manager,
    OUTPUT_DIR / "B4_route_explorer.html",
    title=f"B4 Jeepney Routes ({NUM_ROUTES})",
    route_notes=route_notes,
)
print(html_path)
display(IFrame(html_path.as_uri(), width=1200, height=900))


C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_baseline_routes\B4_route_explorer.html


## 5. Standalone Rejection Sampling Setup
This cell rebuilds the route generator and systemic evaluator so part 2 can run without any earlier notebook state.


In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from tqdm.auto import tqdm

REPO_ROOT = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import BaselineRouteGenerator, SystemicFitnessEvaluator

SAMPLE_SIZE = 5000
GOOD_PERCENTILE = 95.0
ELITE_PERCENTILE = 99.0
RISK_AVERSION_K = 1.0
ROBUSTNESS_CV_CEILING = 0.20
SCREENING_TESTS = 1
VALIDATION_TESTS = 10
SCREENING_BACKGROUND_ROUTE_MEAN = 1
SCREENING_BACKGROUND_ROUTE_STD = 0
VALIDATION_BACKGROUND_ROUTE_MEAN = 2
VALIDATION_BACKGROUND_ROUTE_STD = 1
RANDOM_SEED = 20260422
OUTPUT_YAML = REPO_ROOT / "configs" / "B4_rejection_sampling.yaml"

sampler = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=RANDOM_SEED,
)
screening_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=SCREENING_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=SCREENING_BACKGROUND_ROUTE_MEAN,
    background_route_std=SCREENING_BACKGROUND_ROUTE_STD,
    seed=RANDOM_SEED,
)
validation_evaluator = SystemicFitnessEvaluator(
    passenger_map=sampler.passenger_map,
    drive_graph_raw=sampler.drive_graph_raw,
    drive_graph_proj=sampler.drive_graph_proj,
    evaluation_test_mean=VALIDATION_TESTS,
    evaluation_test_std=0.0,
    background_route_mean=VALIDATION_BACKGROUND_ROUTE_MEAN,
    background_route_std=VALIDATION_BACKGROUND_ROUTE_STD,
    seed=RANDOM_SEED,
)
route_seed_rng = np.random.default_rng(RANDOM_SEED)
validation_seed_rng = np.random.default_rng(RANDOM_SEED + 1)

def risk_adjusted_score(result) -> float:
    return float(-(float(result.average_gtc) + (RISK_AVERSION_K * float(result.std_gtc))))

def coefficient_of_variation(result) -> float:
    mean_gtc = float(result.average_gtc)
    if mean_gtc <= 0:
        return float("inf")
    return float(result.std_gtc / mean_gtc)

print(f"Standalone setup ready: {SAMPLE_SIZE:,} candidates, P{GOOD_PERCENTILE:.0f}/P{ELITE_PERCENTILE:.0f} thresholds")


Standalone setup ready: 10,000 candidates, P95/P99 thresholds


### 5.1 Sample and screen candidate routes
This cell generates 10,000 candidates with a progress bar and scores each one against the systemic evaluator.


In [4]:
screening_rows = []
candidate_by_id = {}

for index in tqdm(range(SAMPLE_SIZE), desc="Sampling candidate routes", unit="route"):
    while True:
        route_seed = int(route_seed_rng.integers(0, np.iinfo(np.int32).max))
        try:
            route = sampler.generate_route(route_id=f"B4Q{index + 1:05d}", seed=route_seed)
            break
        except ValueError:
            continue

    candidate_by_id[route.route_id] = route
    result = screening_evaluator.evaluate(route, n_tests=SCREENING_TESTS, seed=route_seed)
    screening_rows.append(
        {
            "route_id": route.route_id,
            "average_gtc": float(result.average_gtc),
            "std_gtc": float(result.std_gtc),
            "risk_adjusted_score": risk_adjusted_score(result),
            "reward": float(result.reward),
            "average_passenger_gtc_std": float(result.average_passenger_gtc_std),
            "n_tests": int(result.n_tests),
            "area_m2": float(route.polygon_area_m2),
            "length_m": float(route.path_length_m),
            "attempt": int(route.attempt_index),
            "coefficient_of_variation": coefficient_of_variation(result),
        }
    )

screening_scores = pd.DataFrame(screening_rows).sort_values("risk_adjusted_score", ascending=False).reset_index(drop=True)
display(screening_scores.head(10))


Sampling candidate routes:   0%|          | 0/10000 [00:00<?, ?route/s]

KeyboardInterrupt: 

### 5.2 Filter for robust routes
This cell applies percentile cutoffs, the mean-plus-two-sigma check, and the coefficient-of-variation ceiling.


In [ ]:
score_values = screening_scores["risk_adjusted_score"].to_numpy(dtype=float)
mean_gtc = float(screening_scores["average_gtc"].mean())
std_gtc = float(screening_scores["std_gtc"].mean())
mean_risk_adjusted_score = float(score_values.mean())
std_risk_adjusted_score = float(score_values.std(ddof=0))
good_threshold = float(np.percentile(score_values, GOOD_PERCENTILE))
elite_threshold = float(np.percentile(score_values, ELITE_PERCENTILE))
sigma_threshold = float(mean_risk_adjusted_score + 2.0 * std_risk_adjusted_score)
robustness_limit = float(ROBUSTNESS_CV_CEILING)

good_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= good_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
elite_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= elite_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
sigma_scores = screening_scores.loc[(screening_scores["risk_adjusted_score"] >= sigma_threshold) & (screening_scores["coefficient_of_variation"] <= robustness_limit)].copy().reset_index(drop=True)
good_routes = [candidate_by_id[row.route_id] for row in good_scores.itertuples(index=False)]

print(f"Accepted {len(good_routes)} routes at P{GOOD_PERCENTILE:.0f}")
print(f"Risk-adjusted threshold: {good_threshold:.3f}")
print(f"Mean + 2σ threshold: {sigma_threshold:.3f}")
print(f"Robustness ceiling (CV): {robustness_limit:.2f}")
display(good_scores.head(10))


### 5.3 Validate and export
This cell re-evaluates the accepted routes with a broader systemic pass, then saves the run settings to YAML.


In [ ]:
validation_rows = []
for route in tqdm(good_routes, desc="Validating accepted routes", unit="route"):
    validation_seed = int(validation_seed_rng.integers(0, np.iinfo(np.int32).max))
    validation = validation_evaluator.evaluate(route, n_tests=VALIDATION_TESTS, seed=validation_seed)
    validation_rows.append(
        {
            "route_id": route.route_id,
            "average_gtc": float(validation.average_gtc),
            "std_gtc": float(validation.std_gtc),
            "risk_adjusted_score": risk_adjusted_score(validation),
            "reward": float(validation.reward),
            "average_passenger_gtc_std": float(validation.average_passenger_gtc_std),
            "n_tests": int(validation.n_tests),
            "coefficient_of_variation": coefficient_of_variation(validation),
        }
    )

validation_scores = pd.DataFrame(
    validation_rows,
    columns=[
        "route_id",
        "average_gtc",
        "std_gtc",
        "risk_adjusted_score",
        "reward",
        "average_passenger_gtc_std",
        "n_tests",
        "coefficient_of_variation",
    ],
).sort_values("risk_adjusted_score", ascending=False).reset_index(drop=True)

yaml_payload = {
    "experiment": "B4_rejection_sampling",
    "higher_is_better": True,
    "sample_size": SAMPLE_SIZE,
    "random_seed": RANDOM_SEED,
    "risk_aversion_k": RISK_AVERSION_K,
    "robustness_cv_ceiling": ROBUSTNESS_CV_CEILING,
    "route_generator": {
        "min_area_m2": 2_000_000,
        "anchor_pool_size": 96,
        "max_attempts": 500,
        "route_prefix": "B4Q",
    },
    "screening": {
        "good_percentile": GOOD_PERCENTILE,
        "elite_percentile": ELITE_PERCENTILE,
        "tests": SCREENING_TESTS,
        "background_route_mean": SCREENING_BACKGROUND_ROUTE_MEAN,
        "background_route_std": SCREENING_BACKGROUND_ROUTE_STD,
        "percentile_threshold": good_threshold,
        "elite_threshold": elite_threshold,
        "mean_plus_2sigma_threshold": sigma_threshold,
        "robustness_ceiling": robustness_limit,
    },
    "validation": {
        "tests": VALIDATION_TESTS,
        "background_route_mean": VALIDATION_BACKGROUND_ROUTE_MEAN,
        "background_route_std": VALIDATION_BACKGROUND_ROUTE_STD,
    },
    "summary": {
        "mean_gtc": mean_gtc,
        "std_gtc": std_gtc,
        "mean_risk_adjusted_score": mean_risk_adjusted_score,
        "std_risk_adjusted_score": std_risk_adjusted_score,
        "good_route_count": int(len(good_scores)),
        "elite_route_count": int(len(elite_scores)),
        "sigma_route_count": int(len(sigma_scores)),
        "best_route_id": str(screening_scores.iloc[0]["route_id"]),
        "best_risk_adjusted_score": float(screening_scores.iloc[0]["risk_adjusted_score"]),
        "best_average_gtc": float(screening_scores.iloc[0]["average_gtc"]),
        "best_std_gtc": float(screening_scores.iloc[0]["std_gtc"]),
        "validation_mean_gtc": float(validation_scores["average_gtc"].mean()) if not validation_scores.empty else None,
        "validation_std_gtc": float(validation_scores["average_gtc"].std(ddof=0)) if len(validation_scores) > 1 else 0.0 if len(validation_scores) == 1 else None,
        "validation_mean_risk_adjusted_score": float(validation_scores["risk_adjusted_score"].mean()) if not validation_scores.empty else None,
        "validation_std_risk_adjusted_score": float(validation_scores["risk_adjusted_score"].std(ddof=0)) if len(validation_scores) > 1 else 0.0 if len(validation_scores) == 1 else None,
        "validation_best_route_id": str(validation_scores.iloc[0]["route_id"]) if not validation_scores.empty else None,
    },
}
OUTPUT_YAML.write_text(yaml.safe_dump(yaml_payload, sort_keys=False, allow_unicode=False), encoding="utf-8")

print(f"Validation routes: {len(validation_scores)}")
print(f"YAML saved to: {OUTPUT_YAML}")
display(validation_scores.head(10))
